# Template · Bayesian network (pgmpy)
Copy this notebook into `Partial/`, rename it (e.g. `subject1.ipynb`) and replace the example
network A → C ← B with the one in the exam. Steps: structure → CPDs → check → graph → queries → independence.

In [ ]:
%pip install -q pgmpy networkx matplotlib

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import networkx as nx
from pgmpy.models import DiscreteBayesianNetwork      # older pgmpy: BayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

In [ ]:
# 1) Structure: list of (parent, child)
model = DiscreteBayesianNetwork([("A", "C"), ("B", "C")])

# 2) CPDs. Rows = states of the child. Columns = parent combinations, LAST parent changes fastest.
cpd_a = TabularCPD("A", 2, [[0.3], [0.7]], state_names={"A": ["yes", "no"]})
cpd_b = TabularCPD("B", 2, [[0.6], [0.4]], state_names={"B": ["yes", "no"]})
cpd_c = TabularCPD("C", 2,
                   #  A=yes,B=yes  A=yes,B=no  A=no,B=yes  A=no,B=no
                   [[0.9,          0.7,        0.4,        0.1],     # C = yes
                    [0.1,          0.3,        0.6,        0.9]],    # C = no
                   evidence=["A", "B"], evidence_card=[2, 2],
                   state_names={"C": ["yes", "no"], "A": ["yes", "no"], "B": ["yes", "no"]})
model.add_cpds(cpd_a, cpd_b, cpd_c)

# 3) Consistency check (every column sums to 1, CPDs match the graph)
print("Model OK:", model.check_model())

In [ ]:
# 4) Graph -> PNG for the upload. Write the positions by hand for a tidy drawing.
pos = {"A": (-1, 1), "B": (1, 1), "C": (0, 0)}
nx.draw(nx.DiGraph(model.edges()), pos, with_labels=True, node_size=1500,
        node_color="#cfe2f3", arrowsize=20, font_weight="bold")
plt.savefig("network.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5) Inference
infer = VariableElimination(model)
q = infer.query(["A"], evidence={"C": "yes"}, show_progress=False)       # posterior P(A | C = yes)
print(q)
print("P(A = yes | C = yes) =", round(q.get_value(A="yes"), 4))
print("MAP (A, B) | C = yes:", infer.map_query(["A", "B"], evidence={"C": "yes"}, show_progress=False))

In [ ]:
# 6) Independence (True = independent). Justify also path by path in a text cell.
print("A ⊥ B     ?", not model.is_dconnected("A", "B"))                   # collider closed -> True
print("A ⊥ B | C ?", not model.is_dconnected("A", "B", observed=["C"]))   # collider opened -> False
print(model.local_independencies(["A", "B", "C"]))